# Hospital Readmission Prediction

Predict 30-day readmission of diabetic patients (`readmitted == "<30"`).

1. Data loading
2. Target
3. Data cleaning
4. EDA
5. Feature importance
6. Validation framework (split by patient)
7. Models: logistic regression, random forest, XGBoost (with tuning)
8. Final model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit

## 1. Data loading

Missing values are encoded as `?`. The string `None` is **not** missing: for `A1Cresult` and `max_glu_serum` it means "test not performed". pandas treats `None` as NaN by default, so we disable the default NA values.

In [ ]:
df = pd.read_csv(
    "data/diabetic_data.csv",
    na_values="?",
    keep_default_na=False,
    low_memory=False,
)
df.shape

## 2. Target

`readmitted` has three values: `NO`, `>30` and `<30`. We predict readmission within 30 days.

In [ ]:
print(df["readmitted"].value_counts(normalize=True).round(3))

df["target"] = (df["readmitted"] == "<30").astype(int)
df["target"].mean()

## 3. Data cleaning

Patients who died or were discharged to a hospice cannot be readmitted, so they are removed (`discharge_disposition_id` 11, 13, 14, 19, 20, 21, see `data/IDS_mapping.csv`).

TODO:
- missing values per column; decide what to drop (`weight` is ~97% missing) and what to fill with `"NA"`
- `gender == "Unknown/Invalid"` (3 rows)
- group `diag_1`, `diag_2`, `diag_3` ICD-9 codes into a few categories (circulatory, respiratory, diabetes, ...)
- drop columns that are IDs or leak the target (`encounter_id`, `readmitted`)

In [ ]:
EXPIRED_OR_HOSPICE = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(EXPIRED_OR_HOSPICE)].reset_index(drop=True)

df.isna().mean().sort_values(ascending=False).head(10)

## 4. EDA

TODO (2 points need *extensive* EDA):
- ranges of the numerical features (`describe()`, histograms)
- target distribution and class imbalance
- readmission rate per category (age, admission type, `A1Cresult`, insulin, ...)
- correlations between numerical features

## 5. Feature importance

TODO: mutual information for categorical features, ROC AUC of each numerical feature (as in homework 4), risk ratio per category.

## 6. Validation framework

The same patient can have several encounters. A random split would put encounters of the same patient in train and validation, which leaks information. We split **by patient** (`patient_nbr`): 60% train, 20% validation, 20% test.

In [ ]:
def split_by_patient(df, test_size, seed=1):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(df, groups=df["patient_nbr"]))
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)


df_full_train, df_test = split_by_patient(df, test_size=0.2)
df_train, df_val = split_by_patient(df_full_train, test_size=0.25)

len(df_train), len(df_val), len(df_test)

## 7. Models

TODO (3 points need several models **and** tuning), evaluate with ROC AUC on validation:
- logistic regression (`DictVectorizer` + `LogisticRegression`), tune `C`
- random forest, tune `n_estimators`, `max_depth`, `min_samples_leaf`
- XGBoost, tune `eta`, `max_depth`, `min_child_weight`, number of rounds

## 8. Final model

TODO: retrain the best model on `df_full_train`, evaluate once on `df_test`, then move this logic to `train.py`.